# La caché KV global por token en cuatro generaciones de DeepSeek

Cuaderno de lectura de la medición `deepseek-kv/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos). Respalda el artículo [cuatro-capas-de-cuarenta](https://manpla.net/posts/cuatro-capas-de-cuarenta/). Carga el fichero de al lado —o lo descarga del repositorio si se ejecuta fuera de él—, muestra la ficha de procedencia y dibuja una figura con matplotlib a secas. Solo lee; no vuelve a tomar la instantánea: para eso está `generar.py`.

*Reading notebook for this measurement: loads the file next to it, prints the provenance record and draws one figure. Column names are in Spanish; `GLOSARIO.md` gives the English form.*

In [ ]:
import io, json, urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/deepseek-kv/"

def leer(nombre, **kw):
    """el fichero de al lado si existe; si no, el del repositorio."""
    p = Path(nombre)
    if p.exists():
        return pd.read_csv(p, **kw)
    return pd.read_csv(RAW + nombre, **kw)

def texto(nombre):
    p = Path(nombre)
    if p.exists():
        return p.read_text(encoding="utf-8")
    with urllib.request.urlopen(RAW + nombre, timeout=30) as r:
        return r.read().decode("utf-8")


## Ficha de procedencia

In [ ]:
print(texto("INSTANTANEA.md"))

## El dato

In [ ]:
g = leer("generaciones.csv")
print(round(g.bytes_token_ficha.iloc[0] / g.bytes_token_ficha.iloc[-1]), "veces entre la primera generación y la última")
g[["modelo", "fecha", "capas", "capas_kv", "entradas_por_token", "bytes_entrada_implicitos", "bytes_entrada_declarados", "residuo_por_entrada", "bytes_token_ficha"]]

## Una figura

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4.2))
a.bar(g.modelo, g.bytes_token_ficha); a.set_yscale("log"); a.set_ylabel("bytes por token (escala log)"); a.set_title("caché KV global por token, según la ficha")
a.tick_params(axis="x", rotation=20)
b.bar(g.modelo, g.entradas_por_token); b.set_yscale("log"); b.set_ylabel("entradas de caché por token (escala log)"); b.set_title("entradas por token, según el config")
b.tick_params(axis="x", rotation=20); plt.tight_layout()